# SmolLM2-135M Memory Fusion — Sequential Acceptance Training

This experiment uses the new training algorithm:

1. Start from the original **SmolLM2-135M Transformer**.
2. Replace **one attention layer at a time**.
3. Train the new Memory Fusion layer against the original Transformer attention function.
4. Curriculum the calibration input from mostly teacher hidden states to the **real hidden states produced by the partially converted student**.
5. Do **not** replace the next layer until the current layer passes all acceptance gates:
   - function NMSE,
   - function cosine similarity,
   - incremental model-level ΔNLL,
   - cumulative model-level ΔNLL vs the original Transformer.
6. After all 30 replacements are accepted, train the integrated model in three stages:
   - all Memory Fusion cores + output projections,
   - plus RMSNorm parameters,
   - finally the **entire model** with a very small learning rate.

Progress is stored in **Google Drive after every accepted layer**, so a Colab disconnect does not lose the conversion.


In [ ]:
import os, sys, pathlib, subprocess, importlib, json, math, gc, time
import torch

subprocess.run(['nvidia-smi'], check=False)

REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-e', str(REPO_DIR),
    'transformers==4.57.6',
    'datasets>=3,<5',
    'huggingface_hub>=0.34,<2',
    'pandas',
    'matplotlib',
], check=True)

SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())


## Persistent Google Drive checkpoint

The output directory is on Google Drive. If Colab disconnects, reopen this notebook and rerun it with `RESUME=True`; it will continue from the last **accepted** layer instead of starting over.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('persistent root:', DRIVE_ROOT)


## Settings

The default is **Memory Fusion r64**, because r64 was slightly better than r48 in the previous full-model experiment.

`STRICT_ACCEPTANCE=True` means the experiment stops if a layer cannot satisfy the acceptance gates. This is intentional: a bad replacement is never silently allowed to contaminate deeper layers.

For a first scientific run, keep the defaults. If training stops on a particular layer, inspect that layer's metrics before relaxing any threshold.


In [ ]:
BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
MEMORY_RANK = 64
FEATURE_DIM = 32
CONTEXT_LENGTH = 128
SEED = 73
RESUME = True
STRICT_ACCEPTANCE = True

# Per-layer functional training.
MIN_LAYER_STEPS = 50
MAX_LAYER_STEPS = 300
CHECK_EVERY = 25
LAYER_LR = 2e-4

# Teacher -> real-student hidden-state curriculum.
TEACHER_ALPHA_START = 0.90
TEACHER_ALPHA_END = 0.00

# Acceptance gates.
ACCEPT_NMSE = 0.20
ACCEPT_COSINE = 0.90
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05

# Integrated training after all 30 replacements pass.
CORE_O_TOKENS = 50_000
NORM_TOKENS = 50_000
FULL_TOKENS = 100_000

MAX_RUNTIME_MINUTES = 240
OUTPUT_DIR = DRIVE_ROOT / f'smollm2-memory-fusion-sequential-r{MEMORY_RANK}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('output:', OUTPUT_DIR)


## Train / resume the sequential conversion

At each layer the script prints an **ACCEPTANCE CHECK**. A layer passes only when all four conditions are satisfied. Only then is the next Transformer layer removed.

During the layer fit, the loss also includes small full-model KL and CE terms, so the replacement learns not only the local attention mapping but also how to preserve the student's language-model behavior.


In [ ]:
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts' / 'train_smollm2_memory_fusion_sequential.py'),
    '--base-model', BASE_MODEL,
    '--output-dir', str(OUTPUT_DIR),
    '--memory-rank', str(MEMORY_RANK),
    '--feature-dim', str(FEATURE_DIM),
    '--context-length', str(CONTEXT_LENGTH),
    '--seed', str(SEED),
    '--min-layer-steps', str(MIN_LAYER_STEPS),
    '--max-layer-steps', str(MAX_LAYER_STEPS),
    '--check-every', str(CHECK_EVERY),
    '--layer-lr', str(LAYER_LR),
    '--teacher-alpha-start', str(TEACHER_ALPHA_START),
    '--teacher-alpha-end', str(TEACHER_ALPHA_END),
    '--accept-nmse', str(ACCEPT_NMSE),
    '--accept-cosine', str(ACCEPT_COSINE),
    '--accept-incremental-delta-nll', str(ACCEPT_INCREMENTAL_DELTA_NLL),
    '--accept-cumulative-delta-nll', str(ACCEPT_CUMULATIVE_DELTA_NLL),
    '--core-o-tokens', str(CORE_O_TOKENS),
    '--norm-tokens', str(NORM_TOKENS),
    '--full-tokens', str(FULL_TOKENS),
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
]
cmd.append('--resume' if RESUME else '--no-resume')
cmd.append('--strict-acceptance' if STRICT_ACCEPTANCE else '--no-strict-acceptance')

print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Layer-by-layer acceptance report

This table is the key diagnostic. We want later layers to remain close even though their inputs increasingly come from previous Memory Fusion layers.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

report_path = OUTPUT_DIR / 'sequential_training_report.json'
progress_path = OUTPUT_DIR / 'sequential_progress.json'

if report_path.exists():
    report = json.loads(report_path.read_text())
elif progress_path.exists():
    progress = json.loads(progress_path.read_text())
    report = {
        'status': progress.get('stage', 'in_progress'),
        'accepted_layers': progress.get('accepted_layers', []),
        'layer_reports': progress.get('layer_reports', []),
    }
else:
    raise FileNotFoundError('No sequential report/progress file found.')

print('status:', report.get('status'))
print('accepted layers:', len(report.get('accepted_layers', [])))

layer_df = pd.DataFrame(report.get('layer_reports', []))
display(layer_df)

if not layer_df.empty:
    plt.figure(figsize=(12, 4))
    plt.plot(layer_df['layer'], layer_df['nmse'], marker='o', label='NMSE')
    plt.axhline(ACCEPT_NMSE, linestyle='--', label='NMSE threshold')
    plt.xlabel('Layer')
    plt.ylabel('NMSE')
    plt.title('Sequential replacement: functional NMSE')
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(layer_df['layer'], layer_df['cumulative_delta_nll'], marker='o', label='cumulative ΔNLL')
    plt.axhline(ACCEPT_CUMULATIVE_DELTA_NLL, linestyle='--', label='acceptance limit')
    plt.axhline(0.0, linestyle=':')
    plt.xlabel('Layer')
    plt.ylabel('ΔNLL vs original Transformer')
    plt.title('Model-level drift as Transformer layers are replaced')
    plt.legend()
    plt.show()


## Reload the final full-state model

This is different from the earlier checkpoint format: after the final tiny-LR full-model stage, FFNs/norms/QKV can also change. Therefore this experiment saves the **entire model state**, not only the custom attention tensors.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.smollm2_memory_fusion import (
    SmolMemoryFusionConfig,
    replace_all_attention,
    structural_summary,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = (
    torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == 'cuda' else torch.float32)
)

config_path = OUTPUT_DIR / 'smollm2_memory_fusion_sequential_config.json'
state_path = OUTPUT_DIR / 'smollm2_memory_fusion_sequential_full.pt'

if not config_path.exists() or not state_path.exists():
    raise RuntimeError(
        'Final full-state checkpoint is not available yet. '
        'The sequential conversion must accept all 30 layers and reach final saving first.'
    )

meta = json.loads(config_path.read_text())
mf_config = SmolMemoryFusionConfig.from_dict(meta['memory_fusion'])

student = AutoModelForCausalLM.from_pretrained(meta['base_model'], dtype=dtype).to(device)
replace_all_attention(student, mf_config)
full_state = torch.load(state_path, map_location='cpu', weights_only=True)
student.load_state_dict(full_state, strict=True)
student.config.use_cache = False
student.eval()

summary = structural_summary(student)
print(summary)
assert summary['memory_fusion_layers'] == 30
assert summary['transformer_attention_layers'] == 0


## Final held-out evaluation against the original SmolLM2 Transformer

The final comparison uses `Salesforce/wikitext` test data, not the FineWeb-Edu stream used for training. Lower NLL/perplexity is better. **ΔNLL < 0** means the fully converted model beats the original Transformer on this test.


In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    wiki = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
except Exception:
    parquet = hf_hub_download(
        repo_id='Salesforce/wikitext',
        repo_type='dataset',
        filename='wikitext-2-raw-v1/test-00000-of-00001.parquet',
    )
    wiki = load_dataset('parquet', data_files={'test': parquet}, split='test')

stream = '\n'.join(str(x) for x in wiki['text'] if str(x).strip())
ids = tokenizer(stream, add_special_tokens=False)['input_ids']

EVAL_CONTEXTS = [128, 256]
EVAL_BLOCKS = 12

def make_blocks(context):
    blocks = []
    for start in range(0, len(ids) - context + 1, context):
        blocks.append(torch.tensor(ids[start:start+context], dtype=torch.long))
        if len(blocks) >= EVAL_BLOCKS:
            break
    return blocks

@torch.no_grad()
def eval_model(model, blocks):
    model.eval()
    losses = []
    for block in blocks:
        x = block.unsqueeze(0).to(device)
        out = model(input_ids=x, labels=x, use_cache=False, return_dict=True)
        losses.append(float(out.loss.detach().float()))
    nll = sum(losses) / len(losses)
    return nll, math.exp(nll)

rows = []
teacher_eval = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype).to(device)
teacher_eval.config.use_cache = False

for context in EVAL_CONTEXTS:
    blocks = make_blocks(context)
    t_nll, t_ppl = eval_model(teacher_eval, blocks)
    s_nll, s_ppl = eval_model(student, blocks)
    rows.extend([
        {'model':'Transformer original','context':context,'nll':t_nll,'ppl':t_ppl,'delta_nll':0.0},
        {'model':f'Memory Fusion sequential r{MEMORY_RANK}','context':context,'nll':s_nll,'ppl':s_ppl,'delta_nll':s_nll-t_nll},
    ])

eval_df = pd.DataFrame(rows)
display(eval_df)

for context in EVAL_CONTEXTS:
    r = eval_df[(eval_df.context == context) & (eval_df.model != 'Transformer original')].iloc[0]
    print(
        f'context={context}: ΔNLL={r.delta_nll:+.6f}  '
        + ('🏆 beats Transformer' if r.delta_nll < 0 else 'Transformer still better')
    )

result_path = OUTPUT_DIR / 'sequential_final_wikitext_eval.csv'
eval_df.to_csv(result_path, index=False)
print('saved:', result_path)


## What success looks like

The most important result is not only the final NLL. Watch the **cumulative ΔNLL curve while layers are replaced**.

A good conversion should remain approximately flat instead of exploding with depth. If a particular layer cannot pass the gate, that layer is telling us exactly where Memory Fusion still lacks a capability of the original attention.
